In [2]:
!pip install gdown
import gdown
file_id ='1bDVaJ2ioPvfAANy3zRUlmoz8w4q9WNG5'
gdown.download(f'https://drive.google.com/uc?id={file_id}', 'final_data.csv', quiet=False)
import pandas as pd


Downloading...
From: https://drive.google.com/uc?id=1bDVaJ2ioPvfAANy3zRUlmoz8w4q9WNG5
To: /content/final_data.csv
100%|██████████| 381k/381k [00:00<00:00, 92.6MB/s]


In [3]:
df = pd.read_csv('final_data.csv')
df.head()

,country,year,Export (US$ Thousand),Import (US$ Thousand),forex_reserves,gdp (US$),HDI,Inflation,per_capita_income,population,unemployment
0,Afghanistan,2009,8538901.21,1285743.04,4.265889e+09,1.241615e+10,0.460,-6.81,403.411183,28394806.0,11.31
1,Afghanistan,2010,12441090.42,612543.73,5.162441e+09,1.585667e+10,0.472,2.18,505.230193,29185511.0,11.35
2,Afghanistan,2011,15228634.58,557585.57,6.344642e+09,1.780510e+10,0.477,11.80,566.668100,30117411.0,11.05
3,Afghanistan,2012,12221985.18,684285.78,7.152304e+09,1.990733e+10,0.489,6.44,605.752004,31161378.0,11.34
4,Afghanistan,2013,12600306.70,802838.90,7.288703e+09,2.014642e+10,0.496,7.39,597.356158,32269592.0,11.19


In [4]:
print(df.info())
print(df.describe())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3370 entries, 0 to 3369
Data columns (total 11 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   country                3370 non-null   object 
 1   year                   3370 non-null   int64  
 2   Export (US$ Thousand)  3370 non-null   float64
 3   Import (US$ Thousand)  3370 non-null   float64
 4   forex_reserves         3370 non-null   float64
 5   gdp (US$)              3370 non-null   float64
 6   HDI                    3370 non-null   float64
 7   Inflation              3370 non-null   float64
 8   per_capita_income      3370 non-null   float64
 9   population             3370 non-null   float64
 10  unemployment           3370 non-null   float64
dtypes: float64(9), int64(1), object(1)
memory usage: 289.7+ KB
None
              year  Export (US$ Thousand)  Import (US$ Thousand)  \
count  3370.000000           3.370000e+03           3.370000e+03   
mean   2006.

In [6]:
from sklearn.preprocessing import StandardScaler, LabelEncoder

In [7]:
le = LabelEncoder()
df['country_encoded'] = le.fit_transform(df['country'])


In [ ]:
feature_columns = [
    'country_encoded',
    'year',
    'Export (US$ Thousand)',
    'Import (US$ Thousand)',
    'forex_reserves',
    'HDI',
    'Inflation',
    'per_capita_income',
    'population',
    'unemployment'
]

In [ ]:
X = df[feature_columns]
y = df['gdp (US$)']


In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [ ]:
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import RandomizedSearchCV

In [ ]:
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import numpy as np


In [ ]:
rf1 = RandomForestRegressor(n_estimators=100, max_depth=15, random_state=42)
rf1.fit(X_train, y_train)
y_pred_rf = rf1.predict(X_test)


In [ ]:
mae_rf = mean_absolute_error(y_test, y_pred_rf)
mse_rf = mean_squared_error(y_test, y_pred_rf)
rmse_rf = np.sqrt(mse_rf)
r2_rf = r2_score(y_test, y_pred_rf)
n_samples = len(y_test)
n_features = X_train.shape[1]
adjusted_r2_rf = 1 - (1 - r2_rf) * (n_samples - 1) / (n_samples - n_features - 1)
mape_rf = np.mean(np.abs((y_test - y_pred_rf) / y_test)) * 100


print("="*50)
print("RANDOM FOREST REGRESSOR - ALL METRICS")
print("="*50)
print(f"R² Score:                          {r2_rf:.4f} ({r2_rf*100:.2f}%)")
print(f"Adjusted R² Score:                 {adjusted_r2_rf:.4f} ({adjusted_r2_rf*100:.2f}%)")
print(f"MAPE (Mean Absolute Percentage):   {mape_rf:.2f}%")
print(f"MAE (Mean Absolute Error):         ${mae_rf:,.2f}")
print(f"MSE (Mean Squared Error):          ${mse_rf:,.2f}")
print(f"RMSE (Root Mean Squared Error):    ${rmse_rf:,.2f}")
print("="*50)

RANDOM FOREST REGRESSOR - ALL METRICS
R² Score:                          0.9953 (99.53%)
Adjusted R² Score:                 0.9952 (99.52%)
MAPE (Mean Absolute Percentage):   16.92%
MAE (Mean Absolute Error):         $27,035,314,513.74
MSE (Mean Squared Error):          $9,057,702,065,563,654,684,672.00
RMSE (Root Mean Squared Error):    $95,171,960,500.79


# RandomizedSearchCV

In [ ]:

params = {
    "n_estimators": [10,25,50,100,150,200],
    "max_depth" : [3, 5, 10, 15, 20],
    "max_leaf_nodes" : [20, 40, 80]
}

rf = RandomForestRegressor()
clf = RandomizedSearchCV(rf, params, scoring = "r2", cv=3, n_jobs = -1, verbose = 1)


clf.fit(X_train, y_train)


Fitting 3 folds for each of 10 candidates, totalling 30 fits


RandomizedSearchCV(cv=3, estimator=RandomForestRegressor(), n_jobs=-1,
                   param_distributions={'max_depth': [3, 5, 10, 15, 20],
                                        'max_leaf_nodes': [20, 40, 80],
                                        'n_estimators': [10, 25, 50, 100, 150,
                                                         200]},
                   scoring='r2', verbose=1)

In [ ]:
clf.best_estimator_

RandomForestRegressor(max_depth=20, max_leaf_nodes=80, n_estimators=25)

In [ ]:
rf = RandomForestRegressor(max_depth=15, max_leaf_nodes=80, n_estimators=25)
rf.fit(X_train, y_train)

rf.score(X_test, y_test)

0.994859625488077

# Gradient Boosting

In [ ]:
params = {
    "n_estimators": [50,100,150,200],
    "max_depth" : [3, 4, 5, 7],
    "max_leaf_nodes" : [20, 40, 80],
    "learning_rate": [0.1, 0.2, 0.3]
}

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import RandomizedSearchCV



gb = GradientBoostingRegressor()
r = RandomizedSearchCV(gb, params, scoring = "r2", cv=3, n_jobs = -1, verbose = 1)


r.fit(X_train, y_train)

Fitting 3 folds for each of 10 candidates, totalling 30 fits


RandomizedSearchCV(cv=3, estimator=GradientBoostingRegressor(), n_jobs=-1,
                   param_distributions={'learning_rate': [0.1, 0.2, 0.3],
                                        'max_depth': [3, 4, 5, 7],
                                        'max_leaf_nodes': [20, 40, 80],
                                        'n_estimators': [50, 100, 150, 200]},
                   scoring='r2', verbose=1)

In [ ]:
r.best_estimator_

GradientBoostingRegressor(max_depth=4, max_leaf_nodes=20, n_estimators=200)

In [ ]:
gb = GradientBoostingRegressor(max_depth=7, max_leaf_nodes=40, n_estimators=200)
gb.fit(X_train, y_train)

gb.score(X_test, y_test)

0.9980400652035853

In [ ]:
y_pred_gb = gb.predict(X_test)

mae_gb = mean_absolute_error(y_test, y_pred_gb)
mse_gb = mean_squared_error(y_test, y_pred_gb)
rmse_gb = np.sqrt(mse_gb)
r2_gb = r2_score(y_test, y_pred_gb)
n_samples = len(y_test)
n_features = X_train.shape[1]
adjusted_r2_gb = 1 - (1 - r2_gb) * (n_samples - 1) / (n_samples - n_features - 1)
mape_gb = np.mean(np.abs((y_test - y_pred_gb) / y_test)) * 100

print("="*50)
print("GRADIENT BOOSTING REGRESSOR - ALL METRICS")
print("="*50)
print(f"R² Score:                          {r2_gb:.4f} ({r2_gb*100:.2f}%)")
print(f"Adjusted R² Score:                 {adjusted_r2_gb:.4f} ({adjusted_r2_gb*100:.2f}%)")
print(f"MAPE (Mean Absolute Percentage):   {mape_gb:.2f}%")
print(f"MAE (Mean Absolute Error):         ${mae_gb:,.2f}")
print(f"MSE (Mean Squared Error):          ${mse_gb:,.2f}")
print(f"RMSE (Root Mean Squared Error):    ${rmse_gb:,.2f}")
print("="*50)

GRADIENT BOOSTING REGRESSOR - ALL METRICS
R² Score:                          0.9980 (99.80%)
Adjusted R² Score:                 0.9980 (99.80%)
MAPE (Mean Absolute Percentage):   17.01%
MAE (Mean Absolute Error):         $21,424,951,927.55
MSE (Mean Squared Error):          $3,748,621,584,834,777,579,520.00
RMSE (Root Mean Squared Error):    $61,225,987,822.45
